# Topic: Python: Pandas & Data Vectorization

## Definition (30-second explanation)
* **Pandas** is the standard Python library for in-memory data manipulation and analysis, built on top of NumPy.
* **Data Vectorization** is the process of executing operations on entire arrays (Series/DataFrames) at once using optimized C-level code, completely bypassing slow, sequential Python `for` loops.

## Why Interviewers Ask This
* **Performance:** Iterating over millions of rows with `.iterrows()` or `.apply()` will crash production pipelines. Interviewers want to see if you can write fast, scalable code.
* **Tooling Mastery:** Pandas is foundational. If you struggle with basic `groupby` or `merge` operations, interviewers assume you will struggle with complex ML feature engineering.
* **Data Intuition:** Tests your ability to handle real-world messy data (e.g., missing values, mismatched joins) before feeding it to ML models.

## Core Concepts
* **Vectorization & Broadcasting:** Applying a single operation (like `df['A'] * 2`) across an entire column simultaneously.
* **Split-Apply-Combine (`groupby`):** Splitting data by keys, applying an aggregation/transformation, and combining the results back into a DataFrame.
* **Relational Joins (`merge`):** Combining datasets using keys (inner, left, right, outer), analogous to SQL joins.
* **Masking / Boolean Indexing:** Filtering DataFrames using condition arrays (e.g., `df[df['age'] > 18]`).

## When to Use
* Exploratory Data Analysis (EDA) and summary statistics.
* Feature engineering and data cleaning (imputing nulls, one-hot encoding).
* Preprocessing structured datasets before loading them into TensorFlow or PyTorch pipelines.

## Advantages
* Massive execution speedup (often 100x to 1000x faster than standard Python loops).
* Highly readable, declarative syntax.
* Deep integration with the broader Python data science ecosystem (Scikit-Learn, NumPy, Matplotlib).

## Limitations
* **Memory Bound:** Pandas operates entirely in-memory (RAM). It struggles with datasets larger than your machine's available memory (unlike PySpark).
* **Copy Overhead:** Chained operations can sometimes create hidden intermediate copies of the data, leading to memory spikes.

## Common Comparisons
* **Pandas vs. PySpark:** Pandas is for single-node, in-memory processing. PySpark is for distributed processing across a cluster.
* **Vectorization vs. `.apply()`:** Vectorization pushes the loop down to C-level. `.apply()` is often just a disguised Python loop and should be avoided for basic mathematical operations.
* **`merge()` vs. `concat()`:** `merge()` joins datasets horizontally based on common keys. `concat()` stacks dataframes horizontally or vertically based on the index.

## Common Interview Traps
* **The `.apply()` Trap:** Candidates frequently use `df.apply(lambda x: ...)` for conditional logic. Interviewers will penalize this. Use `np.select()` or `np.where()` instead.
* **SettingWithCopyWarning:** Modifying a filtered slice of a DataFrame without using `.copy()` first, leading to unpredictable behavior.
* **Ignoring NaNs:** Performing mathematical operations on columns with missing data without specifying a `fillna()` strategy, which propagates `NaN`s throughout the dataset.

## Python / SQL Syntax
```python
# Vectorized conditional logic (Avoid apply!)
import numpy as np
df['category'] = np.where(df['score'] > 90, 'A', 'B')

# Groupby and aggregate
summary = df.groupby('user_id').agg({
    'purchase_amount': ['sum', 'mean'],
    'last_login': 'max'
}).reset_index()

# Left Join
merged_df = pd.merge(df1, df2, on='user_id', how='left')
```

## 45-Second Interview Answer
"Pandas is my primary tool for in-memory data manipulation and feature engineering. In both interviews and production, I strictly prioritize vectorization—using built-in Pandas or NumPy functions—over Python loops or the `.apply()` method. This ensures transformations that would take minutes execute in milliseconds. My standard workflow involves utilizing `merge` for relational joins, `groupby` for aggregations, and vectorized boolean masking for handling missing data and creating new features."

## Practice Questions:

### Q1:
**Scenario:**
You are preparing a dataset for an ML pricing model. You need to calculate a discounted_price based on user tiers.
- 'premium' users get a 20% discount.
- 'standard' users get a 10% discount.
- 'guest' users get no discount (0%).

*Many candidates write a custom python function and use df.apply(), which is slow.*

**Task: Write the most efficient, fully vectorized Pandas/NumPy code to create the discounted_price column. Do not use .apply(), .map() with a dictionary, or any loops.**

In [15]:
# Data:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'user_id': [1, 2, 3, 4],
    'tier': ['premium', 'standard', 'guest', 'premium'],
    'transaction_amount': [100.0, 50.0, 20.0, 200.0]
})

In [16]:
df

,user_id,tier,transaction_amount
0,1,premium,100.0
1,2,standard,50.0
2,3,guest,20.0
3,4,premium,200.0


In [17]:
import pandas as pd
import numpy as np

# OPTION 1: np.select() - Best practice for 3+ conditions (highly readable)
conditions = [
    df['tier'] == 'premium',
    df['tier'] == 'standard'
]
choices = [
    df['transaction_amount'] * 0.80,
    df['transaction_amount'] * 0.90
]
df['discounted_price'] = np.select(conditions, choices, default=df['transaction_amount'])

# OPTION 2: Nested np.where() - Valid, but gets messy with many conditions
df['discounted_price'] = np.where(
    df['tier'] == 'premium', df['transaction_amount'] * 0.80,
    np.where(df['tier'] == 'standard', df['transaction_amount'] * 0.90, df['transaction_amount'])
)

In [18]:
df

,user_id,tier,transaction_amount,discounted_price
0,1,premium,100.0,80.0
1,2,standard,50.0,45.0
2,3,guest,20.0,20.0
3,4,premium,200.0,160.0


* **Common Mistake 1:** Using `df.apply(lambda x: custom_func(x['tier'], x['transaction_amount']), axis=1)`. This forces Pandas to drop out of optimized C-code and run a slow Python loop for every row.
* **Common Mistake 2:** Using `df.replace()` or `.map()` when the calculation requires multiple columns. `.map()` is only for 1-to-1 static mapping, not dynamic calculations.
* **Likely Follow-up:** "How would your approach change if there were 50 different tiers, each with a different discount percentage stored in a separate database table?" (Answer: I would load the database table as a DataFrame and use a `pd.merge()` to bring the discount rates in, then do a single vectorized multiplication `df['amount'] * df['discount_rate']`).